In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
)

from utils.synthetic.pipeline.bronze_handoff import (
    handoff_final_aligned_observations_to_bronze, 
    run_bronze_handoff_loop,
)

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

---

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

BATCH_SIZE = env_int(
    "SYNTHETIC_BRONZE_HANDOFF_BATCH_SIZE",
    500,
)

IF_EXISTS_FLAG = "replace"

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_BRONZE_HANDOFF_COMPLETE_ONLY",
    True,
)

STOP_ON_FAILURE_FLAG = env_bool(
    "SYNTHETIC_BRONZE_HANDOFF_STOP_ON_FAILURE",
    True,
)

MAX_ITERATIONS = env_optional_int(
    "SYNTHETIC_BRONZE_HANDOFF_MAX_ITERATIONS",
    default=None,
)

FINAL_ALIGNMENT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_ALIGNED_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_final_aligned_stage",
    aliases=("FINAL_ALIGNMENT_SOURCE_TABLE_NAME",),
)

BRONZE_HANDOFF_TARGET_TABLE_NAME = env_str(
    "BRONZE_HANDOFF_OBSERVATIONS_TABLE",
    "bronze_observations_input_stage",
    aliases=("BRONZE_HANDOFF_TARGET_TABLE_NAME",),
)

print("Bronze handoff config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("source table:", FINAL_ALIGNMENT_SOURCE_TABLE_NAME)
print("target table:", BRONZE_HANDOFF_TARGET_TABLE_NAME)
print("batch size:", BATCH_SIZE)
print("max iterations:", MAX_ITERATIONS)

---

In [ ]:
engine = get_engine_from_env()

----


### Loop - Run Mode Selection

In [ ]:
# Script style

# Mode

# "row" | "row_batch" | "full_batch"

RUN_MODE = "row_batch"


In [ ]:
results = run_bronze_handoff_loop(
    engine=engine,
    mode=RUN_MODE,
    batch_size=BATCH_SIZE,
    schema=SCHEMA,
    source_table=FINAL_ALIGNMENT_SOURCE_TABLE_NAME,
    target_table=BRONZE_HANDOFF_TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    complete_only=COMPLETE_ONLY_FLAG,
    max_iterations=MAX_ITERATIONS,
    stop_on_failure=STOP_ON_FAILURE_FLAG,
)

print(results[-1] if results else {"status": "no_run"})

----

#### One Shot Full Run

In [ ]:
result = handoff_final_aligned_observations_to_bronze(
    engine=engine,
    mode="full_batch",
    batch_size=BATCH_SIZE,
    schema=SCHEMA,
    source_table=FINAL_ALIGNMENT_SOURCE_TABLE_NAME,
    target_table=BRONZE_HANDOFF_TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    complete_only=COMPLETE_ONLY_FLAG,
)

print(result)

---

In [ ]:
# Dispose Engine
engine.dispose()